In [4]:
import pandas as pd
import json
import re
import PyPDF2

In [20]:
def extractBooksAndPages(text):
    book_and_pages = {}
    start = re.search(r"Genesis\s*\.\.\.", text)
    if start:
        q = text[start.start():]  
    else:
        q = text

    pattern = r"([1-3]?\s?\w+(?: \w+)?)\s*\.\.\.\s*(\d+)"
    matches = re.findall(pattern, q)

    for book, page in matches:
        book_and_pages[book.strip()] = int(page)

    return book_and_pages

In [48]:
def getBookName(text):

    text = text.strip()
    lines = text.splitlines()
    first_line = lines[0].strip()

    # Format 1: Page <num> BookName
    match = re.match(r'^Page\s+\d+\s+((?:\d\s*)?[A-Z][a-z]*(?:\s+[A-Z][a-z]*)*)', first_line)
    if match:
        book_name = match.group(1).strip()
        return book_name

    # Format 2: BookName Page <num>
    match = re.match(r'^((?:\d\s*)?[A-Z][a-z]*(?:\s+[A-Z][a-z]*)*)\s+[Pp]age\s+\d+', first_line)
    if match:
        book_name = match.group(1).strip()
        return book_name
    
    return None


In [ ]:
def cleanBookText(text):
    
    text = text.strip()
    # Remove everything before {1:1} if it exists
    match = re.search(r'\{1:1\}', text)
    if match:
        text = text[match.start():]
    else:
        match = re.search(r'Page\s+\d{1,3}', text, re.IGNORECASE)
        if match:
            text = text[match.end():]
        else:
            return None  # nothing to clean

    # Remove source line and line breaks
    text = re.sub(r'Downloaded from .*', '', text)
    text = text.replace('\n', ' ')
    
    return text


In [237]:
def groupByVerses(text):
    
    append_text = None

    if '{' not in text: 
        return text, None

    if not text.startswith('{'):
        append_text = text.split('{',1)[0].strip()
    
    verses = text.split('{',1)[1].strip()
    verses = '{' + verses

    pattern = r'\{(\d+:\d+)\}\s*(.*?)(?=\{\d+:\d+\}|$)'
    matches = re.findall(pattern, verses)

    verse_dict = {verse: verse_text.strip() for verse, verse_text in matches}

    return append_text, verse_dict

In [ ]:
reader = PyPDF2.PdfReader('../data/pdf/The-Holy-Bible-King-James-Version.pdf')

In [119]:
contents_table = reader.pages[2] # 2 is where the table of contents is located
contents = extractBooksAndPages(contents_table.extract_text())
with open("../data/books.json", "w") as f:  
    json.dump(contents, f, indent=4)

In [226]:
print(len(reader.pages))

742


In [242]:
startPage = 21
books = {}
current_book = None

# 579 is where NT matthew starts

for i in range(startPage, len(reader.pages)):
    start_of_chapter = False
    text = reader.pages[i].extract_text()

    if "{1:1}" in text:
        start_of_chapter = True

    bookName = getBookName(text)
    if bookName == "Song":
        bookName = "Song of Songs"
        
    if bookName is not None and bookName != "Psalms":
        bookText = cleanBookText(text)
        verses = groupByVerses(bookText)
        
        if start_of_chapter:
            current_book = bookName    
            books[current_book] = verses[1]
        else:
            if verses[0] is not None:
                last_verse = books[current_book]
                last_key = list(last_verse.keys())[-1]
                last_verse[last_key] += " " + verses[0]
            if verses[1] is not None:
                books[current_book].update(verses[1])



In [ ]:
print(json.dumps(books, indent=4, ensure_ascii=False))

In [243]:
with open("../data/verses.json", "w") as f:  
    json.dump(books, f, indent=4)